# Стандарти електронної пошти

Основні стандарти:

[RFC 5322](https://www.rfc-editor.org/rfc/rfc5322.html) визначає формат інтернет повідомлення

[RFC 5321](https://www.rfc-editor.org/rfc/rfc5321.html) протокол SMTP

[RFC 3501](https://www.rfc-editor.org/rfc/rfc3501.html) протокол IMAP4 rev1

[RFC 9051](https://www.rfc-editor.org/info/rfc9051)  протокол IMAP4 rev2

[RFC 1939](https://www.rfc-editor.org/info/rfc1939) протокол POP3

# Бібліотеки роботи з електронною поштою

In [1]:
import sys
sys.version

'3.14.3 (main, Mar  6 2026, 21:21:26) [GCC 13.3.0]'

Стандартна бібліотека містить бібліотеки:
<ul>
    <li><code>mimetypes</code></li>
    <li><code>email</code></li>
    <li><code>smtplib</code></li>
</ul>

## Визначення типу вкладення

Бібліотека [mimetypes](https://docs.python.org/3/library/mimetypes.html) серед іншого містить засоби, що визначають MIME-тип вкладення згідно IANA.

In [2]:
import mimetypes

In [3]:
mimetypes.types_map['.txt']

'text/plain'

Типове використання: визначення типу файлу за допомогою функції 
[guess_file_type](https://docs.python.org/3/library/mimetypes.html#mimetypes.guess_file_type)
для подальшого вкладення файлу в повідомлення.

Другий аргумент результату &mdash; кодування. Фізична наявність файлу функцією не вимагається.

In [4]:
mimetypes.guess_file_type('report.pdf')

('application/pdf', None)

In [5]:
mimetypes.guess_file_type('report.txt')

('text/plain', None)

In [6]:
mimetypes.guess_file_type('report.py')

('text/x-python', None)

In [7]:
mimetypes.guess_file_type('report.cpp')

('text/x-c++src', None)

## Збирання листа

Бібліотека [<code>email</code>](https://docs.python.org/3/library/email.html) роботи з електронною поштою містить модуль [<code>email.message</code>](https://docs.python.org/3/library/email.message.html), що містить клас [<code>EmailMessage</code>](https://docs.python.org/3/library/email.message.html#email.message.EmailMessage), який зображує емейл-повідомлення.

Методи класу забезпечують додавання полів до повідомлення, а також дозволяють парсити наявне повідомлення.

<code>add_attachment</code>  &mdash; додати вкладення

<code>set_content</code>  &mdash; встановити тіло повідомлення

Деякі поля заголовка листа можна встановлювати наче значення ключів словника.

In [8]:
from email.message import EmailMessage

In [9]:
from pathlib import Path
def _attachx(msg:EmailMessage, fname):
    afname = Path(fname).name
    m, _ = mimetypes.guess_type(afname)
    m_type, m_subtype = m.split('/', 1)
    with open(fname, 'rb') as f:
        msg.add_attachment(f.read(),
                           maintype=m_type,
                           subtype=m_subtype,
                           filename=afname)

In [10]:
def generate_email(*, s_to, s_from, subj, body, attachments=()):
    msg = EmailMessage()
    msg['From'] = s_from
    msg['To'] = s_to 
    # msg['CC'] = 'xxxxxxxx@knu.ua'  
    msg['Subject'] = subj

    msg.set_content(body)

    for x in attachments:
        _attachx(msg, x)
    return msg

## Відправлення за протоколом smtp

Бібліотека [smtplib](https://docs.python.org/3/library/smtplib.html) містить засоби роботи за протоколом SMTP (SMTP-клієнт).

Клас [<code>SMTP</code>](https://docs.python.org/3/library/smtplib.html#smtplib.SMTP) реалізує SMTP-з'єднання. За необхідності використання SSL/TLS слід використовувати дуже схожий клас <code>SMTP_SSL</code>. Для уникнення витоків ресурсів і коректного завершення з'єднання слід використовувати інструкцію <code>with</code>.

Алгоритм відправлення пошти протоколом SMTP:

<ol>
    <li>Відкрити з'єднання (створити об'єкт класу, зазначивши ім'я серверу і за потреби порт).</li> 
    <li> Залогінитись (метод <code>login</code>).</li>
    <li> Перевірити успіх логіну (якщо все гаразд, то відповідь починається з цифри 2).</li>
    <li> Підготувати емейл-повідомлення (об'єкт класу <code>EmailMessage</code>).</li>
    <li> Відправити повідомлення (метод  <code>send_message</code>), якщо все гаразд, то відповідь методу буде порожня. За наявності помилок, якщо жодному отримувачу відправити не вдалося, може бути згенеровано виняток. Якщо винятку не було, але помилки були, то відповідь буде непорожнім словником, в якому ключам-адресам відповідають коди помилок 
    (<a href="https://docs.python.org/3/library/smtplib.html#smtplib.SMTP.sendmail" target=_blank rel="noopener noreferrer">нюанси тут</a>).</li>
    <li> По завершенню сесії, якщо не використовувалась інструкція <code>with</code>, SMTP-з'єднання слід розірвати методом <code>quit</code> (повертає результат виконання SMTP-команди <code>QUIT</code>. </li>
</ol>


Відповідний фрагмент коду може виглядати так:

In [11]:
import smtplib

In [ ]:
try:
    with smtplib.SMTP_SSL('smtp.i.ua', port=465) as ms:
        ms.set_debuglevel(0)
        res = ms.login('login', 'password')
        if res[0] != 235:
            print('Error on login')
            input('press any key')
            raise Exception('Incorrect login')
        msg = generate_email(s_to='кому-адреса', s_from='від_кого-адреса', 
                       subj='Тема листа', body='Повідомлення в тілі листа',) # attachments=('звіт1.pdf','звіт2.pdf'))  
        res = ms.send_message(msg)
        if res:  # лист не бу успішно відправлений
            print(res)
            input("press any key")
            raise Exception('non empty res')
        else:
            print("sended ... ")
    
        # .......
    
        res = ms.quit()
        if res[0] != 221:
            print('Error on quit')
            input('press any key')
except Exception as e:
    print(e)

За відправлення пачки листів доцільно рахувати кількість відправлених. SMTP-сервер може обмежувати відправлення великої кількості листів і вимагати почекати скільки-то часу. Або (Google) заборонити відправлення будь-яких листів (у тому числі і з використанням веб-інтерфейсу) на певний період часу (тривалість бану може залежати від кількості порушень за останній період).

## Отримання пошти за протоколом POP3

Використання протоколу POP3 для отримання пошти потребує хоча б мінімальної бази даних, що відслідковує отримані раніше листи, або режиму роботи поштової скриньки, за якого отримані за POP3 листи не зберігаються на сервері.

# Google API

[Інструкція](https://developers.google.com/workspace/gmail/api/quickstart/python) з встановлення офіційних бібліотек під Python.

Enable Google Gmail API 

In [14]:
from google.auth.transport.requests import Request
from google.oauth2.credentials import Credentials
from google_auth_oauthlib.flow import InstalledAppFlow
from googleapiclient.discovery import build
from googleapiclient.errors import HttpError

In [15]:
import base64
import time
from pathlib import Path

In [22]:
# Check https://developers.google.com/gmail/api/auth/scopes for all available scopes
# If modifying these scopes, delete the file token.json.
SCOPES = ['https://www.googleapis.com/auth/gmail.send']
TOKEN_PATH = Path.home().joinpath('myproject57994.json').resolve()   
SECRETS_PATH = Path.home().joinpath('KNUMailer.json').resolve() 

In [23]:
def get_credentials(secret:Path, scopes, token:Path):
    credentials = None
    # The file token.json stores the user's access and refresh tokens, and is
    # created automatically when the authorization flow completes for the first
    # time.
    if token.exists():
        credentials = Credentials.from_authorized_user_file(token, scopes)
    # If there are no (valid) credentials available, let the user log in.
    if not credentials or not credentials.valid:
        if credentials and credentials.expired and credentials.refresh_token:
            credentials.refresh(Request())
        else:
            flow = InstalledAppFlow.from_client_secrets_file(secret, scopes)
            credentials = flow.run_local_server(port=0)
        # Save the credentials for the next run
        with open(token, 'w') as token:
            token.write(credentials.to_json())
    return credentials

In [24]:
def get_service(secret:Path, scopes, token:Path):
    return build('gmail', 'v1', credentials=get_credentials(secret, scopes, token))

In [25]:
def send_message(service, user_id="me", mime_message=None):
    encoded_message = base64.urlsafe_b64encode(mime_message.as_bytes()).decode()
    message = {'raw': encoded_message}
    try:
        send_message = (service.users().messages().send(userId=user_id, body=message).execute())
    except HttpError as error:
        send_message = None
    return send_message

In [27]:
service = get_service(SECRETS_PATH, SCOPES, TOKEN_PATH)

In [28]:
msg = generate_email(s_to='tkarnaukh@knu.ua', s_from='tkarnaukh@knu.ua', 
                   subj='Тема листа', body='Повідомлення в тілі листа')  

In [30]:
send_message(service, user_id="me", mime_message=msg)

{'id': '19db2f47b17b2ef0',
 'threadId': '19db2f47b17b2ef0',
 'labelIds': ['UNREAD', 'SENT', 'INBOX']}